In [1]:
import json
import logging
import warnings
from pathlib import Path
from typing import List
from dataclasses import dataclass, asdict, field
from enum import Enum
import xml.etree.ElementTree as ET


In [2]:
# Configuration
logging.basicConfig(level=logging.INFO, format="%(message)s")
logger = logging.getLogger(__name__)

# Data Structures
class ChunkType(str, Enum):
    TOKEN_WINDOW = "150-token-window"

@dataclass
class Chunk:
    chunk_id: str
    text: str
    chunk_type: ChunkType
    source_file: str
    position: int
    token_count: int
    sentence_indices: List[int] = field(default_factory=list)
    lemmatized: List[str] = field(default_factory=list)  

    def to_dict(self):
        data = asdict(self)
        data["chunk_type"] = self.chunk_type.value
        return data

# PSC chunker
class PSCChunker:
    def __init__(self, window_size=150, overlap=50):
        self.window_size = window_size
        self.overlap = overlap

    def extract_text_from_tei(self, xml_path: Path) -> str:
        try:
            tree = ET.parse(xml_path)
            root = tree.getroot()
            ns = {"tei": "http://www.tei-c.org/ns/1.0"}

            body = root.find(".//tei:body", ns)
            text = "".join(body.itertext()) if body is not None else "".join(root.itertext())

            return " ".join(text.split())

        except Exception as e:
            logger.warning(f"⚠ XML parse error in {xml_path.name}: {e}")
            return xml_path.read_text(encoding="utf-8")

    def chunk_source(self, source_path: Path) -> List[Chunk]:
        text = self.extract_text_from_tei(source_path)
        if not text.strip():
            return []
        
        # Whitespace tokenization
        tokens = text.split()
        
        if len(tokens) < self.window_size:
            return [
                Chunk(
                    chunk_id=f"{source_path.stem}_window_0",
                    text=" ".join(tokens),
                    chunk_type=ChunkType.TOKEN_WINDOW,
                    source_file=source_path.name,
                    position=0,
                    token_count=len(tokens),
                )
            ]

        chunks = []
        step = self.window_size - self.overlap

        for i, position in enumerate(range(0, len(tokens), step)):
            window_tokens = tokens[position : position + self.window_size]
            if not window_tokens:
                break

            chunks.append(
                Chunk(
                    chunk_id=f"{source_path.stem}_window_{i}",
                    text=" ".join(window_tokens),
                    chunk_type=ChunkType.TOKEN_WINDOW,
                    source_file=source_path.name,
                    position=position,
                    token_count=len(window_tokens),
                )
            )

            if position + self.window_size >= len(tokens):
                break

        return chunks

    def chunk_all_sources(self, sources_dir: Path, output_path: Path):
        source_files = sorted(Path(sources_dir).rglob("*.xml"))
        logger.info(f"Found {len(source_files)} sources to chunk\n")

        all_chunks = []
        total_tokens = 0

        for source_file in source_files:
            logger.info(f"Processing {source_file.name}...")
            chunks = self.chunk_source(source_file)

            total_tokens += sum(c.token_count for c in chunks)
            all_chunks.extend(chunks)

            logger.info(f"✓ {len(chunks)} windows")

        total_chunks = len(all_chunks)
        avg_tokens = total_tokens / total_chunks if total_chunks else 0

        output_data = {
            "metadata": {
                "corpus": "PSC",
                "chunking_strategy": f"{self.window_size}-token sliding window with {self.overlap}-token overlap",
                "window_size": self.window_size,
                "overlap": self.overlap,
                "total_sources": len(source_files),
                "total_chunks": total_chunks,
                "avg_tokens_per_chunk": avg_tokens,
                "lemmatized": False,  # PSC not lemmatized
            },
            "chunks": [c.to_dict() for c in all_chunks],
        }

        output_path.write_text(
            json.dumps(output_data, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

        logger.info("\nPSC CHUNKING COMPLETE")
        logger.info(f"Total chunks: {total_chunks}")
        logger.info(f"Average tokens per chunk: {avg_tokens:.2f}\n")

        return all_chunks


# MAIN

def main():

    sources_dir = Path("../data/")
    output_dir = Path("../chunked_sources/")
    output_dir.mkdir(exist_ok=True)


    psc_chunker = PSCChunker(window_size=150, overlap=50)

    psc_chunks = psc_chunker.chunk_all_sources(
        sources_dir, output_dir / "psc_chunks.json"
    )

    logger.info(f"PSC chunks: {len(psc_chunks):,}")
if __name__ == "__main__":
    main()

Found 700 sources to chunk

Processing tlg1443.tlg001.1st1K-grc1.xml...
✓ 91 windows
Processing tlg1443.tlg002.1st1K-grc1.xml...
✓ 199 windows
Processing tlg1622.tlg001.1st1K-grc1.xml...
✓ 20 windows
Processing tlg1205.tlg001.perseus-grc1.xml...
✓ 113 windows
Processing tlg1205.tlg001.perseus-grc2.xml...
✓ 113 windows
Processing tlg1205.tlg002.perseus-grc1.xml...
✓ 88 windows
Processing tlg1725.tlg001.perseus-grc1.xml...
✓ 216 windows
Processing tlg1447.tlg001.1st1K-grc1.xml...
✓ 251 windows
Processing tlg1447.tlg009.1st1K-grc1.xml...
✓ 4 windows
Processing 001_Auctor-incertus_Responsa-ex-libris-Digestorum.xml...
✓ 6 windows
Processing 001_Tertullianus_Ad-uxorem.xml...
✓ 45 windows
Processing 001_Tertullianus_Apologeticus-adversus-gentes.xml...
✓ 236 windows
Processing 001_Tertullianus_De-idololatria.xml...
✓ 74 windows
Processing 001_Tertullianus_De-poenitentia.xml...
✓ 53 windows
Processing 002_Tertullianus_Adversus-Marcionem.xml...
✓ 901 windows
Processing 002_Tertullianus_De-jejuni

In [6]:
def count_tokens_in_chunks(chunks_json_path: Path):
    """Count total tokens in a chunks JSON file"""
    
    data = json.loads(chunks_json_path.read_text(encoding="utf-8"))
    
    chunks = data["chunks"]
    total_tokens = sum(chunk["token_count"] for chunk in chunks)
    total_chunks = len(chunks)
    avg_tokens = total_tokens / total_chunks if total_chunks else 0
   
    print(f"Total chunks: {total_chunks:,}")
    print(f"Total tokens: {total_tokens:,}")
    print(f"Average tokens per chunk: {avg_tokens:.2f}")
    print(f"Min tokens: {min(c['token_count'] for c in chunks)}")
    print(f"Max tokens: {max(c['token_count'] for c in chunks)}")
    
    # Count by chunk type
    from collections import Counter
    chunk_types = Counter(c["chunk_type"] for c in chunks)
    
    print(f"\nBy chunk type:")
    for chunk_type, count in chunk_types.items():
        type_tokens = sum(c["token_count"] for c in chunks if c["chunk_type"] == chunk_type)
        print(f"  {chunk_type}: {count:,} chunks, {type_tokens:,} tokens")
    
    return {
        "total_chunks": total_chunks,
        "total_tokens": total_tokens,
        "avg_tokens": avg_tokens
    }

if __name__ == "__main__":
    
    psc_path = Path("../chunked_sources/psc_chunks.json")
    
    print("PSC CHUNKS:")
    psc_stats = count_tokens_in_chunks(psc_path)
    print("\n")

PSC CHUNKS:
Total chunks: 209,693
Total tokens: 31,419,916
Average tokens per chunk: 149.84
Min tokens: 51
Max tokens: 150

By chunk type:
  150-token-window: 209,693 chunks, 31,419,916 tokens


